<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Plan overnight bike rebalancing

**[C04-M07-L03 · Plan overnight bike rebalancing](https://learning.nextia-ai.com/courses/sql/m07/plan-overnight-bike-rebalancing/)** · C04, SQL and Data Preparation for AI · Module 7, case study 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** turn a month of real bike-share trips into a station table that answers one question: which 20 stations should the night crew restock before tomorrow's morning peak? On the way you find an export defect, bikes that moved without a rider, and a quiet `LAG` mistake. You test a simple ranking against "same as yesterday", once, on a week you kept apart.

| | |
|---|---|
| **Time** | About 2.5 hours with the case-study page. The notebook runs in about 1 minute. |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6. |
| **You should know** | Window functions and half-open date ranges, from [Time and ranking](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/); quarantine, from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/); target timing, from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/); time splits, from [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/). |
| **You do not need** | The `ticket-data` project. This notebook works in its own folder, `bike-rebalancing/`. |
| **Tested** | 2026-10-07, Python 3.14.6 (SQLite 3.50.4) with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m07/plan-overnight-bike-rebalancing/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M07-L03-plan-overnight-bike-rebalancing/plan-overnight-bike-rebalancing-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6, the version that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see another version, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 matplotlib
import numpy as np
import pandas as pd
print("pandas", pd.__version__)

## Setup: get the data

Run the next cell. It makes a folder `bike-rebalancing` with a `data` folder in it, downloads the trips file (28 MB) from the course's labs repository, and checks its checksum (SHA-256). Everything this notebook writes goes into `bike-rebalancing`. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: data/bike_share_toronto_2025-06.csv.gz (27.7 MB)`. If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `bike-rebalancing` (in Colab: the **Files** panel on the left), then run the cell again.

**The data.** Bike Share Toronto Ridership Data, June 2025, from [City of Toronto Open Data](https://open.toronto.ca/dataset/bike-share-toronto-ridership-data/) (Toronto Parking Authority). Contains information licensed under the [Open Government Licence – Toronto](https://open.toronto.ca/open-data-licence/). Nextia Learning changed the city's file from the cp1252 encoding to UTF-8 and compressed it; nothing else changed. The trips are real; the people in the story are made up.

In [ ]:
# Setup: download the case-study data into bike-rebalancing/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/case-studies/data/"
NAME, SHA256 = "bike_share_toronto_2025-06.csv.gz", "aa77c492b88cefd20117e8303861f8068c4574f99c39a75a9682aa7645d7301e"
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into another project. If the cell ran before, start from the parent.
if Path(".nextia-notebook").exists():
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("bike-rebalancing").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

path = WORK / "data" / NAME
if not (path.exists() and sha256(path) == SHA256):
    if (HERE / NAME).exists():  # a copy next to the notebook
        shutil.copy(HERE / NAME, path)
    else:
        try:
            urllib.request.urlretrieve(LABS + NAME, path)
        except OSError:  # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(path), LABS + NAME]).returncode:
                raise SystemExit(f"Could not download {LABS + NAME}. Check your internet connection.")
if sha256(path) != SHA256:
    raise SystemExit(f"{NAME} has the wrong checksum. Delete the folder bike-rebalancing and run again.")
os.chdir(WORK)  # work in the case-study folder
print(f"Ready: data/{NAME} ({path.stat().st_size / 1e6:.1f} MB)")

## Setup: the check helper

The next cell defines `check()`. It compares your answer to a task with the expected one through a short code (a fingerprint), so it does not show the answer. Run the cell. It prints nothing.

In [ ]:
import hashlib, json

def fingerprint(value):
    """A short hash of a value. Floats are rounded to 3 decimals."""
    def norm(v):
        if hasattr(v, "columns"):   # a DataFrame
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):    # a Series, an array or a numpy number
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items())}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        return v if v is None or isinstance(v, (bool, int)) else str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def check(what, yours, expected):
    """Say "Check passed." or what is different, without showing the answer."""
    if yours is ...:
        print(f"Not yet: {what} has no answer. Replace the ... with your code.")
    elif fingerprint(yours) == expected:
        print("Check passed.")
    elif hasattr(yours, "shape"):
        print(f"Not yet: {what} has {yours.shape[0]} rows and {yours.shape[1]} columns; check them, then the values.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected result.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Every night at 22:00, Leila Haddad, who leads operations for a city bike-share system, sends a crew with vans to restock stations before the morning peak. The crew can restock about **20 stations** a night. Her analyst, Tomasz Nowak, must give her the list: the stations that will lose the most bikes between 7:00 and 10:00 tomorrow. Today Leila uses "the stations that emptied this morning". The [lesson page](https://learning.nextia-ai.com/courses/sql/m07/plan-overnight-bike-rebalancing/#the-situation) tells the full story.

## 2. The data

The next cell reads the file into pandas, shows the first row, and copies all rows into a SQLite database, `bikes.db`. `dtype=str` and `keep_default_na=False` keep every value as the text in the file, so an empty value stays `''` and you can count it. Run the cell.

In [ ]:
import sqlite3

raw = pd.read_csv("data/bike_share_toronto_2025-06.csv.gz", dtype=str, keep_default_na=False)
print(raw.shape)
print(raw.iloc[0].to_string())

con = sqlite3.connect("bikes.db")
raw.to_sql("trips_raw", con, if_exists="replace", index=False)

> **Check.** You should see `(980624, 11)` and a first trip that starts at station 7569 at 00:01:56 on 1 June 2025.

**What this tells us.** One row is one rental. Look at the first row closely: it starts at station 7569 and ends at station 7753, but both names are `Toronto Inukshuk Park`. Keep that in mind.

The next cell defines `sql()`, which runs a query and prints the result with its row count (NULL shows as `NULL`), and `scalar()`, which returns one value. Run it. It prints nothing.

In [ ]:
def sql(query, db=None, limit=20):
    """Run a query (on bikes.db, or on `db`) and print the result as a table, with its row count."""
    cursor = (db or con).execute(query)
    rows = cursor.fetchall()
    table = pd.DataFrame(rows[:limit], columns=[d[0] for d in cursor.description], dtype=object)
    print(table.where(table.notna(), "NULL").to_string(index=False))  # show NULL as NULL
    more = f", showing {limit}" if len(rows) > limit else ""
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''}{more})")


def scalar(query):
    """Run a query that returns one value, and return that value."""
    return con.execute(query).fetchone()[0]

### First checks

> **Predict.** Is `Trip_Id` unique? How many stations and bikes do you expect for a big city in one month? Then run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS trips,
       COUNT(DISTINCT Trip_Id) AS trip_ids,
       MIN(Start_Time) AS first_start,
       MAX(Start_Time) AS last_start,
       COUNT(DISTINCT Start_Station_Id) AS stations,
       COUNT(DISTINCT Bike_Id) AS bikes
FROM trips_raw
""")

> **Check.** You should see 980,624 trips and 980,624 trip IDs, from 2025-06-01 00:00:15 to 2025-06-30 23:59:53, 916 stations and 8,874 bikes.

**What this tells us.** `Trip_Id` is unique, so it is the key of the table. The file is one full month, selected by start time.

### The encoding

The city publishes this file in the **cp1252** encoding (Windows), not UTF-8. Our hosted copy was converted to UTF-8. The next cell shows the station names with characters outside plain English letters, and what goes wrong with the wrong encoding. Run it.

In [ ]:
names = pd.read_sql("""
SELECT DISTINCT Start_Station_Name AS name FROM trips_raw
WHERE Start_Station_Name GLOB '*[^ -~]*'   -- a character outside plain ASCII
""", con)["name"]
for name in names:
    print(name, " | read as cp1252:", name.encode("utf-8").decode("cp1252"))

city_bytes = "25 York St \u2013 Union Station".encode("cp1252")  # how the city's file stores the dash
try:
    city_bytes.decode("utf-8")
except UnicodeDecodeError as error:
    print("The city's file read as UTF-8:", error)

> **Check.** You should see `25 York St – Union Station South` and `Lundy Ave / Étienne Brûlé Park`, each with a garbled version, and an error that ends with `invalid start byte`.

**What this tells us.** The wrong encoding fails in one direction and is quiet in the other. A UTF-8 file read as cp1252 gives `â€“` with no error. If you download a fresh file from the city, read it with `encoding="cp1252"`.

### Empty values

Run the cell to count empty values in the columns that the case study needs.

In [ ]:
sql("""
SELECT SUM(End_Station_Id = '') AS no_end_station,
       SUM(End_Time = '') AS no_end_time,
       SUM(End_Station_Id = '' AND End_Time = '') AS both,
       SUM(Bike_Id = '') AS no_bike,
       SUM(Start_Station_Id = '') AS no_start_station
FROM trips_raw
""")

> **Check.** You should see 830 trips without an end station, 130 without an end time, and 106 with neither.

**What this tells us.** About 1 trip in 1,000 has no known end. You cannot count where those bikes arrived.

### The end-station names

Run the cell. It compares the end name with the start name, and the end ID with the start ID.

In [ ]:
sql("""
SELECT COUNT(*) AS trips,
       SUM(End_Station_Name = Start_Station_Name) AS same_name,
       SUM(End_Station_Id = Start_Station_Id) AS same_station_id
FROM trips_raw
""")
sql("""
SELECT Start_Station_Id, End_Station_Id, Start_Station_Name, End_Station_Name
FROM trips_raw
WHERE End_Station_Id <> Start_Station_Id
LIMIT 3
""")

> **Check.** You should see `same_name` 980,624 (every row) and `same_station_id` 42,604.

**What this tells us.** `End_Station_Name` is a copy of `Start_Station_Name` in every row, although only 42,604 trips return to their start station. This is a defect in the export. The end IDs are right (they match real stations), so the case study never uses `End_Station_Name`: it takes every name from the start side.

> **Your turn.** How many trips have a known end station (not `''`) that is **not** the start station, but still carry the start's name as `End_Station_Name`? Put the number in `different_station_same_name` with `scalar()`. Then run the check cell.

In [ ]:
different_station_same_name = ...  # scalar("""SELECT COUNT(*) FROM trips_raw WHERE ...""")
print(different_station_same_name)

In [ ]:
check('different_station_same_name', different_station_same_name, 'e48ad55ef9413dd7')

### One name per station

The next cell builds a `stations` table from the start side, checks that each ID has one name and each name one ID, and lists the end IDs that never appear as a start. Run it.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS stations;
CREATE TABLE stations AS
SELECT Start_Station_Id AS station_id, MIN(Start_Station_Name) AS name
FROM trips_raw
GROUP BY Start_Station_Id;
""")
sql("""
SELECT (SELECT COUNT(*) FROM stations) AS stations,
       (SELECT COUNT(*) FROM (SELECT Start_Station_Id FROM trips_raw
            GROUP BY 1 HAVING COUNT(DISTINCT Start_Station_Name) > 1)) AS ids_with_two_names,
       (SELECT COUNT(*) FROM (SELECT Start_Station_Name FROM trips_raw
            GROUP BY 1 HAVING COUNT(DISTINCT Start_Station_Id) > 1)) AS names_with_two_ids
""")
sql("""
SELECT End_Station_Id, COUNT(*) AS trips
FROM trips_raw
WHERE End_Station_Id <> '' AND End_Station_Id NOT IN (SELECT station_id FROM stations)
GROUP BY End_Station_Id
""")

> **Check.** You should see 916 stations, `0` and `0`, then four end IDs (7638, 7690, 7793, 8065) with 10 trips together.

**What this tells us.** On the start side, ID and name match one to one, so the `stations` table is safe. Four end stations have no name, because nobody started a trip there in June. They are too few to matter.

### Durations

Run the cell. It counts zero, very short and very long trips, and checks that the duration equals the end time minus the start time.

In [ ]:
sql("""
SELECT SUM(CAST(Trip_Duration AS INTEGER) = 0) AS zero_seconds,
       SUM(CAST(Trip_Duration AS INTEGER) < 60) AS under_60_s,
       SUM(CAST(Trip_Duration AS INTEGER) > 86400) AS over_24_h,
       SUM(End_Time <> '' AND ABS((julianday(End_Time) - julianday(Start_Time)) * 86400
           - CAST(Trip_Duration AS INTEGER)) > 2) AS duration_not_end_minus_start
FROM trips_raw
""")

> **Check.** You should see 779 zero-second trips, 2,426 under 60 seconds, 5 over 24 hours and `0` in the last column.

**What this tells us.** The duration agrees with the times in every row, so the times are consistent. The problems are the trips themselves: a trip of 0 seconds did not happen.

## 3. Clean the trips

First, a typed table `trips`: numbers become integers and `''` becomes NULL (unknown), so SQL's NULL rules work. Run the cell.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS trips;
CREATE TABLE trips AS
SELECT CAST(Trip_Id AS INTEGER) AS trip_id,
       CAST(Trip_Duration AS INTEGER) AS duration_s,
       Start_Station_Id AS start_id,
       Start_Time AS start_time,
       NULLIF(End_Station_Id, '') AS end_id,     -- '' becomes NULL: unknown
       NULLIF(End_Time, '') AS end_time,
       Bike_Id AS bike_id,
       User_Type AS user_type
FROM trips_raw;
""")
sql("SELECT COUNT(*) AS trips, COUNT(end_id) AS with_end_station, COUNT(end_time) AS with_end_time FROM trips")

> **Check.** You should see 980,624 trips, 979,794 with an end station and 980,494 with an end time.

Now each trip gets a **reason** if it cannot count as a rider trip, in a fixed order: the first rule that matches wins. A **false start** is a bike taken and put back at the same station within 2 minutes. Trips with a reason go to the `quarantine` table, as in [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/); nothing is deleted. Run the cell.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS checked;
CREATE TABLE checked AS
SELECT *,
       CASE
           WHEN end_time IS NULL THEN 'no_end_time'
           WHEN end_id IS NULL THEN 'no_end_station'
           WHEN duration_s > 86400 THEN 'over_24_hours'
           WHEN end_id = start_id AND duration_s < 120 THEN 'false_start'
           WHEN duration_s < 60 THEN 'under_60_seconds'
       END AS reason
FROM trips;
DROP TABLE IF EXISTS clean;
CREATE TABLE clean AS SELECT * FROM checked WHERE reason IS NULL;
DROP TABLE IF EXISTS quarantine;
CREATE TABLE quarantine AS SELECT * FROM checked WHERE reason IS NOT NULL;
""")
sql("SELECT COALESCE(reason, '(clean)') AS reason, COUNT(*) AS trips FROM checked GROUP BY reason ORDER BY trips DESC")
print("clean + quarantine =", scalar("SELECT COUNT(*) FROM clean") + scalar("SELECT COUNT(*) FROM quarantine"))

> **Check.** You should see 974,870 clean trips, then `false_start` 3,665, `under_60_seconds` 1,233, `no_end_station` 724, `no_end_time` 130, `over_24_hours` 2, and `clean + quarantine = 980624`.

**What this tells us.** 5,754 trips (0.6%) go to quarantine; 974,870 remain. The last line proves that no trip was lost or counted twice.

## 4. Find the bikes that moved without a rider

If a bike's trip ends at station B, its next trip should start at B. If it starts somewhere else, nobody rode it there: a van moved it. `LAG` reads a value from the row before, inside each bike's own trips. First, a tiny example with four trips. Run the cell.

In [ ]:
tiny = sqlite3.connect(":memory:")
tiny.executescript("""
CREATE TABLE trips (trip_id INTEGER, bike_id TEXT, start_time TEXT, start_id TEXT, end_id TEXT);
INSERT INTO trips VALUES
    (1, 'B7', '2025-06-02 07:40', 'A', 'B'),
    (2, 'B7', '2025-06-02 12:10', 'B', 'C'),
    (3, 'B7', '2025-06-03 08:05', 'D', 'B'),
    (4, 'B9', '2025-06-02 09:00', 'C', 'A');
""")
sql("""
SELECT bike_id, trip_id, start_time, start_id, end_id,
       LAG(end_id) OVER (PARTITION BY bike_id ORDER BY start_time, trip_id) AS previous_end
FROM trips
""", db=tiny)

> **Check.** Trip 3 starts at `D`, but `previous_end` is `C`: bike B7 moved from C to D without a trip. Trip 4 belongs to another bike, so its `previous_end` is NULL: the window never crosses from one bike to another.

Now the same on all trips. Run the cell.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS bike_steps;
CREATE TABLE bike_steps AS
SELECT trip_id, bike_id, start_time, start_id, end_id, end_time,
       LAG(end_id) OVER bike_order AS previous_end,
       LAG(end_time) OVER bike_order AS previous_end_time,
       ROW_NUMBER() OVER bike_order AS step
FROM trips
WINDOW bike_order AS (PARTITION BY bike_id ORDER BY start_time, trip_id);
""")
sql("""
SELECT CASE
           WHEN step = 1 THEN 'first trip of the bike'
           WHEN previous_end IS NULL THEN 'previous end unknown'
           WHEN previous_end = start_id THEN 'starts where it was left'
           ELSE 'moved: no rider trip'
       END AS what_happened,
       COUNT(*) AS trips,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM trips), 1) AS pct
FROM bike_steps
GROUP BY what_happened
ORDER BY trips DESC
""")

> **Check.** You should see 910,507 trips that start where the bike was left (92.8%), **60,554 moved (6.2%)**, 8,874 first trips and 689 after a trip with no end station.

**What this tells us.** About 1 trip in 16 starts after a move that no rider made. That is the operator's rebalancing, repairs, or missing records; the file cannot tell which.

> **Your turn.** `bike_steps` also has `previous_end_time`. How many trips start **before** the same bike's previous trip ended? Put the number in `overlapping_trips`. Then run the check cell.

In [ ]:
overlapping_trips = ...  # scalar("""SELECT COUNT(*) FROM bike_steps WHERE ...""")
print(overlapping_trips)

In [ ]:
check('overlapping_trips', overlapping_trips, '44cb730c420480a0')

A bike cannot be on two trips at once, so those trips are record errors. They are rare, so we only note them.

When are moved bikes used next? Run the cell, then the chart cell.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS moves;
CREATE TABLE moves AS
SELECT trip_id, bike_id, previous_end AS from_id, previous_end_time AS last_seen,
       start_id AS to_id, start_time AS next_start
FROM bike_steps
WHERE previous_end IS NOT NULL AND previous_end <> start_id;
""")
moves_by_hour = pd.read_sql("""
SELECT CAST(strftime('%H', next_start) AS INTEGER) AS hour, COUNT(*) AS moves
FROM moves GROUP BY hour ORDER BY hour
""", con)
print(moves_by_hour.sort_values("moves", ascending=False).head(4).to_string(index=False))
morning_share = moves_by_hour.loc[moves_by_hour["hour"].between(6, 9), "moves"].sum() / moves_by_hour["moves"].sum()
print(f"moved bikes next taken from 6:00 to 9:59: {morning_share:.1%}")

> **Check.** You should see 8:00 at the top with 6,465 moves, and 23.7% of the moved bikes next taken from 6:00 to 9:59.

In [ ]:
import matplotlib.pyplot as plt

ax = moves_by_hour.plot(x="hour", y="moves", kind="bar", figsize=(9, 3), legend=False, color="#4c78a8")
ax.set_xlabel("Hour when the moved bike was next taken")
ax.set_ylabel("Moved bikes, June")
plt.show()

**What this tells us.** The crews already place bikes where morning riders take them. Where do they take bikes from, and where to? Run the cell.

In [ ]:
sql("""
SELECT m.from_id AS station_id, s.name, COUNT(*) AS bikes_taken_away
FROM moves AS m JOIN stations AS s ON s.station_id = m.from_id
GROUP BY m.from_id ORDER BY bikes_taken_away DESC LIMIT 5
""")
sql("""
SELECT m.to_id AS station_id, s.name, COUNT(*) AS bikes_brought
FROM moves AS m JOIN stations AS s ON s.station_id = m.to_id
GROUP BY m.to_id ORDER BY bikes_brought DESC LIMIT 5
""")

> **Check.** At the top you should see `King St W / Bay St (West Side)` (3,179 bikes taken away) and `Centre Island Ferry Dock` (2,208 bikes brought).

**What this tells us.** Vans empty downtown stations that fill up, and the island ferry docks. They bring bikes to residential streets and to the island.

## 5. Build the station table

The target is each station's **net outflow** from 7:00 to 10:00: departures minus arrivals. A station with 30 departures and 5 arrivals loses 25 bikes. First, the trap. In a table with one row per station-day **that had a trip**, a quiet morning has no row. Run the tiny example.

In [ ]:
tiny.executescript("""
CREATE TABLE mornings (station_id TEXT, day TEXT, net_out INTEGER);
INSERT INTO mornings VALUES ('A', '2025-06-02', 5), ('A', '2025-06-03', 6), ('A', '2025-06-05', 9);
""")
sql("""
SELECT station_id, day, net_out,
       LAG(day) OVER (PARTITION BY station_id ORDER BY day) AS row_before,
       LAG(net_out) OVER (PARTITION BY station_id ORDER BY day) AS yesterday_wrong
FROM mornings
""", db=tiny)

> **Check.** On 2025-06-05, `yesterday_wrong` is 6, the value of 3 June. There was no row for 4 June, so `LAG` went back two days without a warning.

The fix is a **grid**: every station on every day, with 0 for a quiet morning. Run the cell. The time ranges are half-open, as in [Time and ranking](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/#half-open-ranges).

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS morning;
CREATE TABLE morning AS
WITH days AS (
    SELECT DISTINCT date(start_time) AS day FROM clean
),
grid AS (                -- every station on every day, so that no day is missing
    SELECT s.station_id, d.day FROM stations AS s CROSS JOIN days AS d
),
departures AS (
    SELECT start_id AS station_id, date(start_time) AS day, COUNT(*) AS n FROM clean
    WHERE time(start_time) >= '07:00:00' AND time(start_time) < '10:00:00'
    GROUP BY 1, 2
),
arrivals AS (
    SELECT end_id AS station_id, date(end_time) AS day, COUNT(*) AS n FROM clean
    WHERE time(end_time) >= '07:00:00' AND time(end_time) < '10:00:00'
    GROUP BY 1, 2
)
SELECT g.station_id, g.day,
       CAST(strftime('%w', g.day) AS INTEGER) BETWEEN 1 AND 5 AS working,
       COALESCE(d.n, 0) AS departures,
       COALESCE(a.n, 0) AS arrivals,
       COALESCE(d.n, 0) - COALESCE(a.n, 0) AS net_out
FROM grid AS g
LEFT JOIN departures AS d ON d.station_id = g.station_id AND d.day = g.day
LEFT JOIN arrivals AS a ON a.station_id = g.station_id AND a.day = g.day;
""")
sql("""
SELECT COUNT(*) AS station_days, COUNT(DISTINCT station_id) AS stations, COUNT(DISTINCT day) AS days,
       SUM(departures + arrivals = 0) AS quiet_mornings, SUM(departures) AS departures, SUM(arrivals) AS arrivals
FROM morning
""")

> **Check.** You should see 27,480 station-days (916 × 30), 6,777 quiet mornings, 129,217 departures and 124,284 arrivals.

How often would the table without the grid have fooled `LAG`? Run the cell.

In [ ]:
sql("""
WITH sparse AS (        -- the table without the grid: only mornings with a trip
    SELECT * FROM morning WHERE departures + arrivals > 0
),
stepped AS (
    SELECT day, LAG(day) OVER (PARTITION BY station_id ORDER BY day) AS row_before
    FROM sparse
)
SELECT COUNT(*) AS rows_with_a_row_before,
       SUM(julianday(day) - julianday(row_before) > 1) AS row_before_is_not_yesterday
FROM stepped WHERE row_before IS NOT NULL
""")

> **Check.** You should see 19,798 rows with a row before, and 1,766 where that row is not yesterday.

**What this tells us.** Without the grid, 1,766 values called "yesterday" would come from an older day, and no error would say so.

## 6. Explore what the decision needs

We explore weeks 1 to 3 only: week 4 is the test. Run the cell.

In [ ]:
sql("""
WITH per_morning AS (
    SELECT day, working, SUM(MAX(net_out, 0)) AS drained_bikes, SUM(net_out > 0) AS draining_stations
    FROM morning
    WHERE day < '2025-06-23'          -- explore weeks 1 to 3 only: week 4 is the test
    GROUP BY day
)
SELECT CASE working WHEN 1 THEN 'working day' ELSE 'weekend' END AS morning,
       COUNT(*) AS mornings,
       ROUND(AVG(drained_bikes)) AS drained_bikes,
       ROUND(AVG(draining_stations)) AS draining_stations
FROM per_morning GROUP BY working
""")

> **Check.** You should see about 2,153 bikes lost per working morning at 407 stations, against 898 bikes at 322 stations on a weekend morning.

**What this tells us.** The morning drain is a working-day problem, so the station table keeps working days only. Is it spread out, or concentrated? Run the cell.

In [ ]:
sql("""
WITH ranked AS (
    SELECT day, net_out, ROW_NUMBER() OVER (PARTITION BY day ORDER BY net_out DESC, station_id) AS rank
    FROM morning WHERE working = 1 AND day < '2025-06-23'
)
SELECT ROUND(1.0 * SUM(CASE WHEN rank <= 20 THEN net_out END) / SUM(MAX(net_out, 0)), 3) AS top_20_share
FROM ranked
""")
sql("""
WITH moved_in AS (       -- moved bikes that a rider took before 10:00 on a working day
    SELECT to_id AS station_id, COUNT(*) AS bikes FROM moves
    WHERE time(next_start) < '10:00:00' AND next_start < '2025-06-23'
      AND CAST(strftime('%w', next_start) AS INTEGER) BETWEEN 1 AND 5
    GROUP BY to_id
)
SELECT m.station_id, s.name, ROUND(AVG(m.net_out), 1) AS mean_net_out,
       ROUND(COALESCE(MAX(v.bikes), 0) / 15.0, 1) AS moved_in_per_morning
FROM morning AS m
JOIN stations AS s USING (station_id)
LEFT JOIN moved_in AS v USING (station_id)
WHERE m.working = 1 AND m.day < '2025-06-23'
GROUP BY m.station_id ORDER BY mean_net_out DESC LIMIT 8
""")

> **Check.** You should see `top_20_share` 0.194, then `Fort York  Blvd / Capreol Ct` at the top with 27.6 bikes lost per working morning and 0.3 moved bikes.

**What this tells us.** The 20 stations that drain most hold about a fifth of the drain, so a list of 20 is worth making. Some top stations already get many moved bikes (800 Fleet St: 21.7); the top two get almost none.

## 7. Features from earlier days only

At 22:00, today's morning is known; tomorrow's is not. Each feature uses earlier rows only. Two kinds of window: over **every day** (`yesterday`, `same_day_last_week`) and over **working days only** (`last_working_day`, `mean_5_working`). The frame `ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING` stops one row before the current one. Run the cell.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS station_day;
CREATE TABLE station_day AS
WITH calendar AS (       -- windows over every day: one row back = the day before
    SELECT *,
           LAG(net_out, 1) OVER (PARTITION BY station_id ORDER BY day) AS yesterday,
           LAG(net_out, 7) OVER (PARTITION BY station_id ORDER BY day) AS same_day_last_week
    FROM morning
)
SELECT station_id, day, net_out, yesterday, same_day_last_week,
       LAG(net_out, 1) OVER w AS last_working_day,
       AVG(net_out) OVER (w ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS mean_5_working,
       COUNT(net_out) OVER (w ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS history_days
FROM calendar
WHERE working = 1        -- windows over working days only: one row back = the working day before
WINDOW w AS (PARTITION BY station_id ORDER BY day);
""")
sql("""
SELECT day, net_out, yesterday, same_day_last_week, last_working_day,
       ROUND(mean_5_working, 1) AS mean_5_working, history_days
FROM station_day WHERE station_id = '7000' AND day >= '2025-06-02' AND day < '2025-06-11'
""")

> **Check.** On Monday 2025-06-09, `yesterday` is 0 (a Sunday), `last_working_day` is 18 (Friday) and `mean_5_working` is 27.2. The real value is 19.

**What this tells us.** "Same as yesterday" compares a Monday with a Sunday. The other features do not have that problem.

## 8. Split and baselines

Week 1 gives history only. We choose the rule on weeks 2 and 3, and test it once on week 4. Run the cell.

In [ ]:
VALID = ("2025-06-09", "2025-06-21")   # weeks 2 and 3: choose the rule here
TEST = ("2025-06-23", "2025-07-01")    # week 4 (23 to 30 June): used once, at the end
sql("""
SELECT CASE WHEN day < '2025-06-09' THEN 'week 1 (history only)'
            WHEN day < '2025-06-21' THEN 'weeks 2-3 (choose)'
            ELSE 'week 4 (test)' END AS part,
       COUNT(DISTINCT day) AS mornings, MIN(day) AS first, MAX(day) AS last,
       SUM(history_days < 5) AS rows_without_5_days_of_history
FROM station_day GROUP BY part ORDER BY first
""")

> **Check.** You should see 5, 10 and 6 mornings, and 4,580 rows without 5 days of history, all in week 1.

The next cell defines `coverage()`. Each morning it ranks the stations by a rule, takes the top 20, and adds up the bikes they really lost. It compares that with the **best possible** 20 stations, which nobody can know in advance. Ties are broken by `station_id`, so the result is always the same. Run it. It prints nothing.

In [ ]:
K = 20   # stations the night crew can restock


def coverage(rule, period, k=K, table="station_day"):
    """Rank the stations each morning by `rule` (a column), take the top k, and
    compare the bikes they lost 7:00-10:00 with the best possible top k."""
    if not (rule.isidentifier() and table.isidentifier()):
        raise ValueError("rule and table must be column and table names")
    query = f"""
        WITH ranked AS (
            SELECT day, net_out,
                   ROW_NUMBER() OVER (PARTITION BY day ORDER BY {rule} DESC, station_id) AS plan_rank,
                   ROW_NUMBER() OVER (PARTITION BY day ORDER BY net_out DESC, station_id) AS best_rank
            FROM {table} WHERE day >= :first AND day < :end
        )
        SELECT COUNT(DISTINCT day) AS mornings,
               SUM(CASE WHEN plan_rank <= :k THEN MAX(net_out, 0) ELSE 0 END) AS covered,
               SUM(CASE WHEN best_rank <= :k THEN MAX(net_out, 0) ELSE 0 END) AS best,
               SUM(plan_rank <= :k AND best_rank <= :k) AS hits
        FROM ranked"""
    row = con.execute(query, {"first": period[0], "end": period[1], "k": k}).fetchone()
    mornings, covered, best, hits = row
    return {"rule": rule, "mornings": mornings, "covered": covered, "best": best,
            "share": round(covered / best, 3), "hits_per_morning": round(hits / mornings, 1)}

> **Predict.** Which of the four rules covers the largest share of the best possible on weeks 2 and 3? Then run the cell.

In [ ]:
RULES = ["yesterday", "last_working_day", "same_day_last_week", "mean_5_working"]
valid_results = pd.DataFrame([coverage(rule, VALID) for rule in RULES])
print(valid_results.to_string(index=False))

> **Check.** You should see shares of 0.781 (`yesterday`), 0.824, 0.840 and **0.867** (`mean_5_working`).

In [ ]:
ax = valid_results.plot(x="rule", y="share", kind="barh", figsize=(7, 2.6), legend=False, xlim=(0, 1), color="#4c78a8")
ax.set_xlabel("Share of the best possible bikes covered, weeks 2-3")
ax.set_ylabel("")
plt.show()

**What this tells us.** The mean of the last five working days wins: 3,655 bikes against 3,292 for "same as yesterday" over 10 mornings, about 36 more bikes a morning. An average smooths out one odd day.

> **Your turn.** Is a shorter memory better? Make the column `mean_3_working`: the mean of the **3** working days before, as a window over `station_day`. The cell then runs `coverage()` on weeks 2 and 3. Then run the check cell.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS station_day_3;
CREATE TABLE station_day_3 AS
SELECT *,
       NULL AS mean_3_working   -- replace NULL with the window: the mean of the 3 working days before
FROM station_day;
""")
mean_3_result = coverage("mean_3_working", VALID, table="station_day_3")
print(mean_3_result)

In [ ]:
check('mean_3_result', mean_3_result, '2382913ed7284345')

Three days cover 0.833, less than five. Keep five.

### Failure case: a feature that knows the answer

Two tempting features: each station's **June mean**, and a mean of the last five days that **includes today**. Run the cell.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS station_day_leaky;
CREATE TABLE station_day_leaky AS
SELECT *,
       AVG(net_out) OVER (PARTITION BY station_id) AS june_mean,
       AVG(net_out) OVER (PARTITION BY station_id ORDER BY day
                          ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS mean_5_with_today
FROM station_day;
""")
leaky = pd.DataFrame([coverage(rule, VALID, table="station_day_leaky")
                      for rule in ["mean_5_working", "june_mean", "mean_5_with_today"]])
print(leaky.to_string(index=False))

> **Check.** You should see 0.867 for the honest rule, 0.888 for `june_mean` and 0.915 for `mean_5_with_today`.

**What this tells us.** Both "beat" the honest rule because they contain the answer: the June mean includes the target mornings, and `CURRENT ROW` includes the target itself. At 22:00 neither is known. This is **leakage**, as in [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/). A score that looks too good is a reason to check the feature's timing.

## 9. The final test, once

The rule is chosen: `mean_5_working`, top 20. Now week 4. Run the cell.

In [ ]:
test_results = pd.DataFrame([coverage(rule, TEST) for rule in RULES])
print(test_results.to_string(index=False))

> **Check.** You should see 0.722 for `yesterday` and **0.833** for `mean_5_working`: 1,999 bikes against 1,732 over 6 mornings.

Where does the gain come from? Run the cell, then the chart cell.

In [ ]:
def next_day(day):
    return (pd.Timestamp(day) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


test_days = pd.read_sql("SELECT DISTINCT day FROM station_day WHERE day >= :first AND day < :end",
                        con, params={"first": TEST[0], "end": TEST[1]})["day"]
by_day = pd.DataFrame([{"day": day, "weekday": pd.Timestamp(day).day_name()[:3],
                        **{rule: coverage(rule, (day, next_day(day)))["share"]
                           for rule in ["yesterday", "last_working_day", "mean_5_working"]}}
                       for day in test_days])
print(by_day.to_string(index=False))

> **Check.** On Monday 23 June: 0.407 for `yesterday` and 0.818 for `mean_5_working`. On Monday 30 June, `mean_5_working` (0.698) is only just above `last_working_day` (0.694).

In [ ]:
ax = by_day.plot(x="day", y=["yesterday", "last_working_day", "mean_5_working"], marker="o", figsize=(8, 3), ylim=(0.4, 1))
ax.set_ylabel("Share of the best possible")
ax.set_xlabel("Morning, week 4")
plt.show()

**What this tells us.** Most of the gain over "same as yesterday" comes from Mondays. Against the fairer baseline, "the last working day", the mean wins on all 6 mornings, but on two of them by less than 0.01. Six mornings are little evidence, but weeks 2 and 3 pointed the same way.

> **Your turn: change one thing.** Leila may get a third van and restock 40 stations. Make `k40_results`: a DataFrame with `coverage()` of `"yesterday"` and `"mean_5_working"` on `TEST` with `k=40`, in that order. Then run the check cell.

In [ ]:
k40_results = ...  # pd.DataFrame([coverage(rule, TEST, k=40) for rule in [...]])
print(k40_results)

In [ ]:
check('k40_results', k40_results, '9aa6d0a7db7f679f')

## 10. Checks before delivery

Each rule is a query that must return no rows, as in [Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/). Run the cell. Every line should start with `ok`.

In [ ]:
rules = {
    "a station-day appears twice":
        "SELECT station_id, day FROM morning GROUP BY 1, 2 HAVING COUNT(*) > 1",
    "the grid is not stations x days":
        "SELECT 1 WHERE (SELECT COUNT(*) FROM morning) <> "
        "(SELECT COUNT(*) FROM stations) * (SELECT COUNT(DISTINCT day) FROM morning)",
    "a trip is lost or counted twice":
        "SELECT 1 WHERE (SELECT COUNT(*) FROM clean) + (SELECT COUNT(*) FROM quarantine) <> (SELECT COUNT(*) FROM trips_raw)",
    "net_out is not departures - arrivals":
        "SELECT * FROM morning WHERE net_out <> departures - arrivals",
    "an evaluated row has less than 5 days of history":
        "SELECT * FROM station_day WHERE day >= '2025-06-09' AND history_days < 5",
    "a weekend morning is in the station table":
        "SELECT * FROM station_day WHERE CAST(strftime('%w', day) AS INTEGER) NOT BETWEEN 1 AND 5",
}
for name, query in rules.items():
    bad = len(con.execute(query).fetchall())
    print(f"{'FAIL' if bad else 'ok  '}  {name}" + (f": {bad} rows" if bad else ""))

## 11. Deliver the table and the night plan

The next working morning after the data is Wednesday 2 July (1 July is Canada Day, a public holiday). The cell writes the plan for it from the last 5 working days, and the whole station table. Run it.

In [ ]:
plan = pd.read_sql("""
WITH last_5 AS (         -- the 5 working days before Wednesday 2 July (1 July is Canada Day)
    SELECT DISTINCT day FROM station_day ORDER BY day DESC LIMIT 5
)
SELECT m.station_id, s.name, ROUND(AVG(m.net_out), 1) AS expected_net_out
FROM station_day AS m JOIN stations AS s USING (station_id)
WHERE m.day IN (SELECT day FROM last_5)
GROUP BY m.station_id ORDER BY expected_net_out DESC, m.station_id LIMIT 20
""", con)
plan.insert(0, "rank", range(1, len(plan) + 1))
plan.to_csv("night_plan_2025-07-02.csv", index=False)
pd.read_sql("SELECT * FROM station_day ORDER BY day, station_id", con).to_csv("station_day.csv", index=False)
print(plan.head(8).to_string(index=False))
print("station_day.csv:", scalar("SELECT COUNT(*) FROM station_day"), "rows")

> **Check.** You should see `Fort York  Blvd / Capreol Ct` first, with 28.2, and `station_day.csv: 19236 rows`. The folder `bike-rebalancing` now has both files.

**What this tells us.** This is the list for the crew, and the table behind it. The [lesson page](https://learning.nextia-ai.com/courses/sql/m07/plan-overnight-bike-rebalancing/#recommendations) gives the recommendations, the limits and how to turn this into a portfolio piece.

## 12. Recap

- `End_Station_Name` was a copy of the start name in every row; the names now come from the start side.
- 5,754 trips went to quarantine, with a reason each; 60,554 moves without a rider were found with `LAG`.
- A grid of every station and day kept `LAG` honest; the features use earlier rows only.
- The mean of the last 5 working days covered 0.833 of the best possible on the test week, against 0.722 for "same as yesterday". Most of that gain is on Mondays.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/sql/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m07/plan-overnight-bike-rebalancing/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The trips are real (Bike Share Toronto, Open Government Licence – Toronto); the people in the story are made up.